## Topic: LangGraph Core Components

### Agenda:

- 1. Introduction to LangGraph Core Components

- 2. LLM Workflow

- 3. Why Does LangGraph Use Graphs?


- 4.  State — The Shared Memory of a Graph

- 5.  StateGraph — The Graph Builder

- 6.  Nodes — Units of Work

- 7.  Edges — Connections Between Nodes

- 8.  START and END Nodes

- 9.  Reducers — How State Updates Are Merged

- 10. LangGraph Execution Model

- 11. Complete LangGraph Architecture

- 12. Key Takeaways


### 1. Introduction to LangGraph Core Components

- Definition of LangGraph:
    - LangGraph is an orchestration framework for building intelligent, stateful, multi-step and controllable applications of LLM workflow.


    - It enables advanced features like parallelism, loops, branching, memory and resumability __ making it ideal for agentic and production-grade AI applications.

    - It models our logic as a graph of nodes(tasks) and edges(routing) instead of a linear chian.



- Key Rule of Thump:
    - LangGraph Core Components are the building blocks you use to construct graph-based agent workflows.

    - LangChain Components (Bricks):
        - ChatOpenAI, PromptTemplate, Retriever, Tools, Parsers

    - LangGraph Components (Blueprint):
    - State, StateGraph, Nodes, Edges, Conditional Edges, Reducers, Checkpointers, Compile


    - Together:
        - LangGraph blueprint orchestrates LangChain bricks.


### 2. LLM Workflow

- 1. Workflow:
- =================
- 
    - Series of tasks for execute of right order to achieve goal.

- 2. LLM Workflow:
=================
- 
    - LLM workflow are a step by step process using which we can build complex LLM application.

    - Each step in a workflow performs a distinct task __ such as prompting, reasoning, tool calling, memory access, or decision-making.

    - Workflows can be linear, parallel, branched, or looped, allowing for complex behaviours like retries, multi-agent communication, or tool-augmented reasoning.

    - **Each application has a unique workflow**.
    
    - Key Rule of Thump:    
        - If a workflow for execute of tasks use LLM that is called LLM Workflow.


- Common Workflows
- =================
- 
     - 1. Prompt Chaining:
        - We use(call) multiple time LLM in series format.
        - eg: topic --> LLM --> outline --> LLM --> details report
        - use when we have complex task that divide into sub task.

    - 2. Routing:
        - eg:
        -  Input(query) --> LLM call(router)
        -                            ---> LLM call 1
        -                            ---> LLM call 2           ----> output
        -                            ---> LLM call 3

        - router call decide, which LLM is execute.
    
    - 3. Parallelization:
        - eg: Youtube content Checker
        -  Input(content) ---> LLM call 1 (check the youtube community guideline) 
        -                 ---> LLM call 2 (check the misinformation)                    ---> Aggregator ----> output
        -                 ---> LLM call 3 (check the sexual content)          
       
        - Based on the query at a time execute all  LLM.
        - here, the sub task are predefine that each LLM are execute.

    - 4. orchestrator Workers
        - eg: Research Report Generator

        -  Input(query) --> LLM call(orchestrator)
        -                    ---> LLM call 1             ---> Synthesizer ----> output
        -                    ---> LLM call 2           
        -                    ---> LLM call 2      


        - orchestrator LLM automatically decide which LLM is execute at run time.
        - here, we don't know the nature of sub task that are LLM execute.
        - Depending on the query, the orchestrator LLM decides which LLM should be called.


    - 5. Evaluator Optimizer:
        - There are two LLM. first LLM (Generator LLM) are use to Create Generator content and the second LLM (Evaluator LLM) are use for Evaluate of the first LLM response and also provide the Feedback.

### 3. Why Does LangGraph Use Graphs?

In [ ]:
"""
- 1. The Problem with Linear Chains:
=====================================

LangChain Chain (Linear):

  Step A → Step B → Step C → Done

    - Cannot go back to Step A
    - Cannot skip Step B based on a condition
    - Cannot run Step A and Step B in parallel
    - Cannot loop until a condition is met

    
- 2. The Solution: Graphs:
============================
LangGraph (Graph):

         START
           │
           ▼
      ┌─────────┐
      │ Node A  │
      └────┬────┘
           │
      ┌────▼────┐
      │ Router  │ ← Conditional: which path?
      └──┬───┬──┘
     Yes │   │ No
         ▼   ▼
    ┌──────┐ ┌──────┐
    │Node B│ │Node C│
    └──┬───┘ └──┬───┘
       │        │
       └───┬────┘
           ▼
      ┌─────────┐
      │ Node D  │
      └────┬────┘
           │
           ▼
          END

- Key Note: 
    - Can branch (conditional edges)
    - Can loop (edges back to earlier nodes)
    - Can run in parallel (multiple edges from one node)
    - Can merge (multiple edges into one node)


"""

In [ ]:
""" 
Visual Graph Representation: 
============================

                 ┌───────────┐
                 │   START   │
                 └─────┬─────┘
                       │
                       ▼
          ┌────────────────────────┐
          │    create_greeting     │
          │         NODE           │
          └───────────┬────────────┘
                      │
                      ▼
                 ┌───────────┐
                 │    END    │
                 └───────────┘

=======================================================================
                 
                 
                  ┌─────────────┐
                  │    START    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    NODE     │
                  │  Process    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    STATE    │
                  │   Update    │
                  └──────┬──────┘
                         ↓
                  ┌─────────────┐
                  │    EDGE     │
                  │   Routing   │
                  └──────┬──────┘
                         ↓
                    ┌───────┐
                    │  END  │
                    └───────┘

                    
- Key Rule of Thump:
    - Graph = State + Nodes + Edges
    - the entire system is called graph.

    - The System generates an essay topic, collects the student's submission, and evaluates it in parallel on depth of analysis, language quality and clarity of thought . based on the combined score, it either gives feedback for improvement or approves the essay.

"""

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│        WHY LEARN LANGGRAPH COMPONENTS FIRST?                │
│                                                             │
│  1. EVERY WORKFLOW USES THE SAME PARTS                      │
│     Sequential, Parallel, Conditional, Iterative: all are   │
│     just different arrangements of State, Nodes, and Edges. │
│                                                             │
│  2. MOST BUGS COME FROM STATE MISUNDERSTANDING              │
│     "Why was my list overwritten?" → You needed a reducer.  │
│                                                             │
│  3. ROUTING LOGIC IS THE HEART OF AGENTS                    │
│     Conditional edges are how agents "decide."              │
│                                                             │
│  4. PERSISTENCE, MEMORY, AND HITL ALL DEPEND ON CHECKPOINTS │
│     Module 3 and 4 topics build directly on these.          │
│                                                             │
│  5. YOU CAN READ ANY LANGGRAPH CODE                         │
│     Once you know the 10 components, every example in the   │
│     docs becomes readable.                                  │
└─────────────────────────────────────────────────────────────┘


""" 

### 4.  State — The Shared Memory of a Graph

- Definition:
    - State is a shared data object that flows through every node in the graph. Every node can read the state and write updates to it.

    - In LangGraph, state is the shard memory that flows through our workflow __ it holds all the data being passed between nodes as our graph runs.


In [ ]:
""" 
- Example of State:
========================
    State = The "whiteboard" that all nodes share.

        - Node A reads the whiteboard, does work, writes results.
        - Node B reads the updated whiteboard, does work, writes results.
        - Node C reads the final whiteboard.

"""

#### Three Ways to Define State
- 1. TypedDict 
- 2. Pydantic
- 3. MessagesState 

In [3]:
# Problem or Task: To doing two number addition
## Option 1. TypeDict  
from typing import TypedDict

class AddNumber(TypedDict):
    # This data are shareable
    num1 : int
    num2 : int
    result : int



In [ ]:
# Problem : To doing two number addition
## Option 2. Pydantic  
from pydantic import BaseModel, Field

class AddNumber(BaseModel):
    num1 : int = Field(default=0, ge=0, description="Number 1")   # Validates: must be >= 0
    num2 : int = Field(default=0, description= "Number 2")
    result : int = Field(default=0, description="Summation of Number 1 and Number 2")



In [ ]:
# ---------- Option 3: MessagesState (prebuilt, for chatbots) ----------

from langgraph.graph import MessagesState

class ChatState(MessagesState):
    # Already includes: messages: Annotated[list, add_messages]
    user_name: str          # Add your own extra fields

In [ ]:
""" 
┌─────────────────────────────────────────────────────────────┐
│                 CHOOSING A STATE TYPE                       │
│                                                             │
│  TypedDict      → Default choice. Fast, simple, flexible.   │
│                   No runtime validation.                    │
│                                                             │
│  Pydantic       → When you need validation or defaults.     │
│                   Slightly slower.                          │
│                                                             │
│  MessagesState  → Chatbots and agents. Comes with a         │
│                   "messages" list and the right reducer.    │
│                                                             │
│  RULE: Start with TypedDict. Use MessagesState for chat.    │
└─────────────────────────────────────────────────────────────┘

"""

### 5.  Nodes — Units of Work

- Definition:
    - Nodes are Python functions that do the actual work. Each node receives the current state, performs some computation, and returns state updates.

    - A node can call an LLM, call a tool, query a database, or run plain Python.

In [ ]:
""" 
Node = A worker in the factory.

  Input:  Current state (reads from the whiteboard)
  Work:   Calls LLM, searches DB, processes data, etc.
  Output: State updates (writes to the whiteboard)

"""

In [ ]:
""" 
- Node Representation:
=======================

                 START
                   │
                   ▼
            ┌─────────────┐
            │   Node A    │
            └──────┬──────┘
                   │
                   ▼
            ┌─────────────┐
            │   Node B    │
            └──────┬──────┘
                   │
                   ▼
                  END

==================================

                         STATE SCHEMA
                              │
                              ▼
                           STATE
                              │
                              ▼
                    ┌─────────────────┐
                    │      START      │
                    └────────┬────────┘
                             │
                             ▼
                    ┌─────────────────┐
                    │      NODE       │
                    │                 │
                    │   Do some work  │
                    └────────┬────────┘
                             │
                             ▼
                       STATE UPDATE
                             │
                             ▼
                    ┌─────────────────┐
                    │      EDGE       │
                    └────────┬────────┘
                             │
                    ┌────────┴────────┐
                    │                 │
                    ▼                 ▼
              Normal Route      Conditional Route
                    │                 │
                    └────────┬────────┘
                             ▼
                         Next Node
                             │
                             ▼
                           ...
                             │
                             ▼
                    ┌─────────────────┐
                    │       END       │
                    └─────────────────┘


"""

In [4]:
## Example of Node
# A node is just a Python function with this signature:

def my_addition_node(state: AddNumber) -> AddNumber:
    """
    Args:
        state: The current state of the graph (full state dict)

    Returns:
        update state : Perform the state operation and Update the current state
    """
    # Read from state
    num1 = state["num1"]
    num2 = state["num2"]

    # perform addition operation
    result = num1 + num2


    # Return updates (NOT the full state)
    return state["return"]

In [7]:
# Adding Nodes to a Graph
from langgraph.graph import StateGraph

builder = StateGraph(AddNumber)
builder.add_node("my_addition_node", my_addition_node)       


- Key NOTE:
    - The Graph is Created

In [ ]:
""" 
        ┌─────────────────────────────────────────────────────────────┐
        │              NODE RULES                                     │
        │                                                             │
        │  1. A node is ANY callable: function, class, lambda         │
        │                                                             │
        │  2. Input:  Receives the FULL current state                 │
        │                                                             │
        │  3. Output: Returns a DICT of state updates                 │
        │     Only include fields that changed.                       │
        │                                                             │
        │  4. Nodes can call ANYTHING:                                │
        │     • LangChain LLMs, chains, retrievers                    │
        │     • External APIs, databases, file systems                │
        │     • Other Python libraries                                │
        │                                                             │
        │  5. Nodes should be PURE (no side effects ideally)          │
        │     Given the same state, produce the same output.          │
        │                                                             │
        │  6. Node names must be UNIQUE within a graph                │
        └─────────────────────────────────────────────────────────────┘

"""

### 6.  Edges — Connections Between Nodes
- Definition:
    - Edges define where the workflow goes next. They connect nodes together to form the execution path.

    - A normal edge is a fixed connection: after A, always run B.


In [ ]:
""" 
- Types of Edges:
==================

┌─────────────────────────────────────────────────────────────┐
│              THREE TYPES OF EDGES                           │
│                                                             │
│  1. NORMAL EDGE (Fixed)                                     │
│     graph.add_edge("A", "B")                                │
│     After A, ALWAYS go to B.                                │
│                                                             │
│     A ──────→ B                                             │
│                                                             │
│  2. CONDITIONAL EDGE (Dynamic)                              │
│     graph.add_conditional_edges("A", router_fn, {...})      │
│     After A, go to B or C depending on state.               │
│                                                             │
│         ┌──→ B  (if condition X)                            │
│     A ──┤                                                   │
│         └──→ C  (if condition Y)                            │
│                                                             │
│  3. ENTRY/EXIT EDGES                                        │
│     graph.add_edge(START, "A")   ← Where to begin           │
│     graph.add_edge("C", END)     ← Where to stop            │
│                                                             │
│     START ──→ A ──→ ... ──→ C ──→ END                       │
└─────────────────────────────────────────────────────────────┘

"""

In [8]:
# Edges Connector 
from langgraph.graph import StateGraph, START, END

# Normal edges (fixed path)
builder.add_edge(START, "my_addition_node")      # Start at step_1
builder.add_edge("my_addition_node", END)        # step_3 → End

In [ ]:
""" 
- Visual:
===========
  ┌───────┐       ┌────────────────────┐       ┌──────┐
  │ START │ ────→ │  my_addition_node  │ ────→ │  END │ 
  └───────┘       └────────────────────┘       └──────┘
  (entry)                                          (exit)

"""

In [ ]:
""" 
    - STATE, NODES, EDGES and GRAPH: 
    ==================================
    
            ┌──────────────────────────────────────────────────────────────┐
            │                                                              │
            │  STATE                                                       │
            │  Shared data that moves through the workflow                 │
            │  Example: messages, user query, retrieved documents          │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  NODES                                                       │
            │  Python functions that perform work                          │
            │  Example: call_llm(), search_documents(), grade_answer()     │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  EDGES                                                       │
            │  Define where the workflow goes next                         │
            │  Example: agent → tools → agent                              │
            │                                                              │
            │       │                                                      │
            │       ▼                                                      │
            │  GRAPH                                                       │
            │  The complete workflow created with StateGraph               │
            │                                                              │
            └──────────────────────────────────────────────────────────────┘

"""

### 5.  StateGraph — The Graph Builder

### 8.  Reducers — How State Updates Are Merged

- Definition:
    - Reducers in LangGraph define how updates from nodes are applied to the shared state.

    - Each key in the state can have it's own reducer, which determines whether new data replace, merges or adds to the existing  value.
    
    - Reducers control how state updates are merged when a node returns new values. By default, LangGraph overwrites the old value. Reducers let you customize this behavior.



- Key points
    - state are accessible and mutable.
    - without Reducers the state values are always modify.
    -  Reducers tell us, how to do the state value (it can update, replace, add, merge)

In [ ]:
""" 
Default (Overwrite):
  State: {"count": 5}
  Node returns: {"count": 10}
  Result: {"count": 10}    ← Old value replaced

With Reducer (Add):
  State: {"count": 5}
  Node returns: {"count": 10}
  Result: {"count": 15}    ← Values added together

"""

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│               THE 10 LANGGRAPH CORE COMPONENTS                   │
│                                                                  │
│  DATA                                                            │
│    1. State            → Shared data schema (TypedDict/Pydantic) │
│    2. Reducers         → Rules for merging state updates         │
│                                                                  │
│  WORK                                                            │
│    3. Nodes            → Python functions that do the work       │
│                                                                  │
│  FLOW                                                            │
│    4. Edges            → Fixed connections (A → B)               │
│    5. Conditional Edges→ Dynamic routing (A → B or C)            │
│    6. START / END      → Entry and exit points                   │
│                                                                  │
│  BUILD & RUN                                                     │
│    7. StateGraph       → The builder; compile() makes it runnable│
│    8. Execution        → invoke(), stream(), batch()             │
│                                                                  │
│  PRODUCTION                                                      │
│    9. Checkpointer     → Persistence, memory, resume, rewind     │
│   10. Send / Command / interrupt → Map-reduce, routing, HITL     │
└──────────────────────────────────────────────────────────────────┘


"""

### 9.  START and END Nodes

- Define where execution begins and terminates

### 10. LangGraph Execution Model

- 1. Graph Definition:
    - The state schema.
    - Nodes(functions that Perform tasks).
    - Edges (which node connects to which).

- 2. Compilation:
    - call the .compile() on the StateGraph.
    - This checks the graph structure and prepares it for execution.

- 3. Invocation:
    - run the graph with .invoke(initial_state)
    - LangGraph sends the initial state as a message to the entry nodes.

- 4. Super-Steps Begin:
    - Execution Proceeds in rounds.


- 5. Message Passing and Node Activation:
    - The messages are passed to downstream nodes via edges.
    - Nodes that receive messages become active for the next round.

- 6. Halting Condition:
    - Execution stop when
        - No nodes are active and 
        - No messages are in transit.

### 11. Complete LangGraph Architecture

In [ ]:
""" 
                         LANGGRAPH APPLICATION
                                  │
                                  ▼
                         ┌─────────────────┐
                         │  State Schema   │
                         │                 │
                         │ Defines State   │
                         └────────┬────────┘
                                  │
                                  ▼
                         ┌─────────────────┐
                         │      State      │
                         │                 │
                         │ Current Context │
                         │ Data / Messages │
                         └────────┬────────┘
                                  │
                                  ▼
             ┌────────────────────────────────────────┐
             │                 GRAPH                  │
             │                                        │
             │   ┌─────────┐       ┌─────────┐        │
             │   │ Node A  │──────►│ Node B  │        │
             │   └─────────┘       └────┬────┘        │
             │                          │             │
             │                          ▼             │
             │                    ┌───────────┐       │
             │                    │ Condition │       │
             │                    └─────┬─────┘       │
             │                     ┌────┴────┐        │
             │                     ▼         ▼        │
             │                  Node C     Node D     │
             │                     │         │        │
             │                     └────┬────┘        │
             │                          ▼             │
             └──────────────────────────┼─────────────┘
                                        │
                                        ▼
                                  ┌───────────┐
                                  │    END    │
                                  └───────────┘

"""

### 12. Key Takeaways

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│              LANGGRAPH CORE COMPONENTS                           │
│                                                                  │
│  WHAT:  The building blocks for constructing graph-based         │
│         agent workflows: State, Nodes, Edges, Graph.             │
│                                                                  │
│  THE CORE FORMULA:                                               │
│  ─────────────────                                               │
│  LangGraph App = State + Nodes + Edges + Compile                 │
│                                                                  │
│  THE 8 CORE COMPONENTS:                                          │
│  ──────────────────────                                          │
│                                                                  │
│  1. STATE                                                        │
│     Shared data object (TypedDict) that flows through the graph. │
│     Nodes read it, nodes update it.                              │
│                                                                  │
│  2. StateGraph                                                   │
│     The builder class. You add nodes and edges to it.            │
│     graph = StateGraph(MyState)                                  │
│                                                                  │
│  3. NODES                                                        │
│     Python functions that do work.                               │
│     Input: full state. Output: partial state updates.            │
│     graph.add_node("name", function)                             │
│                                                                  │
│  4. EDGES                                                        │
│     Connections between nodes. Define execution order.           │
│     graph.add_edge("A", "B")                                     │
│                                                                  │
│  5. START / END                                                  │
│     Special sentinel nodes marking entry and exit.               │
│     graph.add_edge(START, "first_node")                          │
│     graph.add_edge("last_node", END)                             │
│                                                                  │
│  6. CONDITIONAL EDGES                                            │
│     Dynamic routing based on state. The heart of agent logic.    │
│     graph.add_conditional_edges("node", router_fn, mapping)      │
│                                                                  │
│  7. REDUCERS                                                     │
│     Control how state updates are merged.                        │
│     add_messages: appends to list instead of overwriting.        │
│     Annotated[list, add_messages]                                │
│                                                                  │
│  8. CHECKPOINTERS                                                │
│     Save state at every step for persistence and memory.         │
│     MemorySaver (dev), SqliteSaver (local), PostgresSaver (prod) │
│                                                                  │
│  EXECUTION:                                                      │
│  ──────────                                                      │
│  app = graph.compile(checkpointer=memory)                        │
│  app.invoke(input, config)    → Full result                      │
│  app.stream(input, config)    → Step-by-step                     │
│  app.batch(inputs, config)    → Parallel                         │
│                                                                  │
│  CONFIG:                                                         │
│  ───────                                                         │
│  config = {"configurable": {"thread_id": "unique-id"}}           │
│  Same graph, different threads = different conversations.        │
│                                                                  │
│  LANGCHAIN vs LANGGRAPH COMPONENTS:                              │
│  ──────────────────────────────────                              │
│  LangChain: LLMs, Prompts, Retrievers, Tools, Parsers            │
│             (What work to do)                                    │
│  LangGraph: State, Nodes, Edges, Reducers, Checkpointers         │
│             (In what order to do it)                             │
│                                                                  │
│  RELATIONSHIP:                                                   │
│  LangChain components go INSIDE LangGraph nodes.                 │
│  LangGraph orchestrates WHEN and HOW they run.                   │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Define your state first. Then your nodes. Then your edges.     │
│   Then compile. The graph IS your application."                  │
└──────────────────────────────────────────────────────────────────┘

"""